In [2]:
!pip install pyspark

from pyspark.sql import SparkSession

# Αρχικοποίηση Spark
spark = SparkSession.builder.master("local[*]").appName("MovieLensProject").getOrCreate()
sc = spark.sparkContext

**1.** Find the top 20 most used words in the dataset. Take into account only the movie titles. If the movie title contains numbers or years, these count as words as well.

In [5]:
# Διάβασμα αρχείου
arxeio = sc.textFile("u.item.csv")
header = arxeio.first()
dedomena = arxeio.filter(lambda line: line != header)

# Απομόνωση τίτλου
grammes = dedomena.map(lambda line: line.split("|"))
titloi = grammes.map(lambda x: x[1])

# Word Count
words = titloi.flatMap(lambda line: line.split(" "))
word_pairs = words.map(lambda word: (word.lower().strip(), 1))
counts = word_pairs.reduceByKey(lambda x, y: x + y)

# Ταξινόμηση
changeKey = counts.map(lambda row: (-row[1], row[0]))
top_20 = changeKey.takeOrdered(20)

# Εκτύπωση
for (plithos, word) in top_20:
    print("%s: %i" % (word, -plithos))

the: 501
(1996): 298
(1995): 296
(1994): 237
(1997): 235
of: 139
(1993): 130
a: 81
in: 75
and: 66
(1998): 53
to: 41
(1992): 40
love: 27
(1990): 24
(1991): 24
man: 23
my: 22
&: 21
for: 20


**2.** Count the males and the females in the dataset.

In [8]:
#Διάβασμα αρχείου u.user.csv
arxeio = sc.textFile("u.user.csv")
header = arxeio.first()
dedomena = arxeio.filter(lambda line: line != header)

#Απομόνωση του φύλου δηλαδή την τρίτη στήλη
grammes = dedomena.map(lambda line: line.split("|"))
gender = grammes.map(lambda x: x[2])

#Καταμέτρηση
pairs = gender.map(lambda g: (g, 1))
counts = pairs.reduceByKey(lambda x, y: x + y)

#Εκτύπωση
apotelesma = counts.collect()

for (gender, plithos) in apotelesma:
    print("%s: %i" % (gender, plithos))

M: 670
F: 273


**3.** Calculate the average number of movies per month. The output should be two columns as follows: month, avg_num_movies where month will have values from January to December.

In [10]:
#Διάβασμα αρχείου
arxeio = sc.textFile("u.item.csv")
header = arxeio.first()
movies = arxeio.filter(lambda line: line != header)

#Χωρίζουμε με το "|" και κρατάμε την ημερομηνία
grammes = movies.map(lambda line: line.split("|"))
movies_with_date = grammes.filter(lambda x: len(x[2]) > 0)

#Εξαγωγή Μήνα και Χρονιάς
def get_month_year(x):
    date_parts = x[2].split("-")
    month = date_parts[1]
    year = date_parts[2]
    return ((month, year), 1)

month_year_pairs = movies_with_date.map(get_month_year)

#Βρίσκουμε πόσες ταινίες βγήκαν κάθε μήνα κάθε χρονιάς
movies_per_month_year = month_year_pairs.reduceByKey(lambda x, y: x + y)

#Κρατάμε τον Μήνα με το πλήθος
month_with_count = movies_per_month_year.map(lambda x: (x[0][0], x[1]))

#Βρίσκουμε τον Μέσο Όρο
avg_per_month = month_with_count.groupByKey().mapValues(lambda x: sum(x)/len(x))

#Εκτύπωση
results = avg_per_month.collect()

#Σειρά των μηνών
month_order = {
    "Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4, "May": 5, "Jun": 6,
    "Jul": 7, "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12
}

#Ταξινόμηση
results.sort(key=lambda x: month_order[x[0]])

print("month avg_num_movies")

for (month, avg) in results:
    print("%-15s %i" % (month, avg))

month avg_num_movies
Jan             16
Feb             16
Mar             12
Apr             15
May             19
Jun             16
Jul             16
Aug             16
Sep             9
Oct             10
Nov             29
Dec             9


**4.** Calculate the average age of the users

In [11]:
#Διάβασμα αρχείου u.user.csv
arxeio = sc.textFile("u.user.csv")
header = arxeio.first()
users = arxeio.filter(lambda line: line != header)

#Χωρισμός με το "|" και απομόνωση της ηλικίας
grammes = users.map(lambda line: line.split("|"))
age = grammes.map(lambda x: int(x[1]))

#Υπολογισμός του πλήθους των χρηστών
total_users = age.count()

#Υπολογισμός του αθροίσματος όλων των ηλικιών
total_age = age.reduce(lambda x, y: x + y)

#Εκτύπωση του τελικού Μέσου Όρου
average_age = total_age / total_users
print("Average age of users: %s" % average_age)

Average age of users: 34.05196182396607


**5.** Count the distinct/unique number of females/males for the genres Horror, Action, Comedy

In [12]:
#Διαβασμα αρχειων
arxeio_users = sc.textFile("u.user.csv")
header_users = arxeio_users.first()
dedomena_users = arxeio_users.filter(lambda line: line != header_users)

arxeio_data = sc.textFile("u.data.csv")
header_data = arxeio_data.first()
dedomena_data = arxeio_data.filter(lambda line: line != header_data)

arxeio_movies = sc.textFile("u.item.csv")
header_movies = arxeio_movies.first()
dedomena_movies = arxeio_movies.filter(lambda line: line != header_movies)

#Δημιουργια ζευγαριων
pairs_users = dedomena_users.map(lambda line: line.split("|")).map(lambda x: (x[0], x[2]))
pairs_data = dedomena_data.map(lambda line: line.split("\t")).map(lambda x: (x[0], x[1]))

#Ενωνουμε data και users με βαση το user_id
enosi_1 = pairs_data.join(pairs_users)

#Ενωση με τις ταινιες
neo_kleidi = enosi_1.map(lambda x: (x[1][0], (x[0], x[1][1])))

#Γραμμη ταινιας
pairs_movies = dedomena_movies.map(lambda line: line.split("|")).map(lambda x: (x[0], x))

#Ενωση με βαση το movie_id
enosi_2 = neo_kleidi.join(pairs_movies)

#Ειδη και ταινιες
def extract_genres(record):
    user_id = record[1][0][0]
    gender = record[1][0][1]
    movie_row = record[1][1]

    results = []

    if movie_row[6] == "1":
        results.append(("Action", gender, user_id))

    if movie_row[10] == "1":
        results.append(("Comedy", gender, user_id))

    if movie_row[16] == "1":
        results.append(("Horror", gender, user_id))

    return results

grammes_genre = enosi_2.flatMap(extract_genres)

#Μοναδικοι χρηστες
monadikoi = grammes_genre.distinct()

pairs = monadikoi.map(lambda x: ((x[0], x[1]), 1))

counts = pairs.reduceByKey(lambda x, y: x + y)

#Εκτυπωση
results = counts.collect()
results.sort()

print("Genre    Gender Unique Users")

for keys, plithos in results:
    genre = keys[0]
    gender = keys[1]
    print("%-8s %s      %i" % (genre, gender, plithos))

Genre    Gender Unique Users
Action   F      271
Action   M      667
Comedy   F      272
Comedy   M      668
Horror   F      228
Horror   M      561
